# Viewing Experience Analysis

This analysis looks at how different viewing experiences relate to audience ratings for movies and TV shows.

The main questions are:

1. Which viewing experiences are associated with stronger audience ratings?
2. Which experiences show up more often among the best-received movies and TV shows?
3. Which other well-rated titles strongly match those experiences?

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

DB_PATH = Path("../../data/warehouse/movie_experience.duckdb")
SQL_DIR = Path("../sql")

con = duckdb.connect(str(DB_PATH))

def load_sql(filename):
    return (SQL_DIR / filename).read_text(encoding="utf-8")

# Recreate the analysis view so this notebook can run on its own.
con.execute(load_sql("01_analysis_dataset.sql"))


## 1. Audience Response Profile

TMDB vote average is used as the main reception measure. Vote count is used to judge how much audience evidence supports each rating and to reduce the influence of titles with very few votes.


In [2]:
# Check the rating and vote-count distributions before setting analysis thresholds.

response_profile = con.sql(
    load_sql("02_audience_response_profile.sql")
).df()

response_profile


,media_type,title_count,avg_rating,median_rating,min_rating,max_rating,avg_vote_count,median_vote_count,vote_count_25th,vote_count_75th,zero_ratings,zero_votes,under_10_votes
0,movie,6553,6.51,6.58,0.0,10.0,2615.0,1185.0,475.0,2936.0,14.0,15.0,73.0
1,tv,1168,7.31,7.47,0.0,10.0,1143.0,424.0,147.0,1132.0,5.0,5.0,64.0


Titles with fewer than 10 votes are excluded from the reception analyses. Movies and TV shows are analyzed separately because their rating and vote-count distributions are different.


## 2. Viewing Experience and Audience Reception

For each experience, titles in the top 25% of experience scores within their media type are compared with the remaining titles.

Two rating measures are kept:

- **Title-level rating lift:** every title contributes equally.
- **Vote-weighted rating lift:** titles with more audience votes contribute more.

Keeping both makes it easier to see whether a pattern holds after accounting for audience volume.


In [3]:
# Compare strong experience titles with the rest of the same media type.

rating_lift = con.sql(
    load_sql("03_experience_rating_lift.sql")
).df()

rating_lift.groupby("media_type").head(10)


,media_type,experience,category,strong_title_count,other_title_count,strong_avg_rating,other_avg_rating,strong_weighted_rating,other_weighted_rating,title_rating_lift,weighted_rating_lift
0,movie,Unforgettable,After-effect,1600.0,4880.0,7.20,6.32,7.71,6.73,0.88,0.98
1,movie,Full of life,Overall experience,1591.0,4889.0,7.12,6.34,7.64,6.76,0.78,0.88
2,movie,Immersive,Overall experience,1597.0,4883.0,7.01,6.38,7.50,6.78,0.63,0.72
3,movie,Intense,Overall experience,1596.0,4884.0,6.99,6.39,7.53,6.82,0.60,0.71
4,movie,Life-affirming,Overall experience,1595.0,4885.0,7.03,6.37,7.54,6.84,0.66,0.70
5,movie,Epic / grand,Story / world experience,1596.0,4884.0,7.03,6.37,7.47,6.78,0.66,0.69
6,movie,Fulfilled,After-effect,1600.0,4880.0,6.98,6.39,7.50,6.84,0.59,0.66
7,movie,Nostalgic,Connection,1611.0,4869.0,7.08,6.35,7.48,6.87,0.73,0.61
8,movie,Provokes curiosity,Thinking / engagement,1590.0,4890.0,6.94,6.40,7.46,6.86,0.54,0.60
9,movie,Cathartic,After-effect,1601.0,4879.0,6.95,6.40,7.45,6.88,0.55,0.57


### Key Findings

The strongest movie relationships remain positive after weighting by vote volume. For example, Immersive, Intense, Life-affirming, Epic / grand, and Nostalgic all show meaningful positive weighted rating lifts.

TV shows also show positive relationships for experiences such as Unforgettable, Full of life, Seasonal, Strong chemistry, Epic / grand, and Rewatchable, although the weighted lifts are generally smaller than the strongest movie results.

These are associations, not evidence that an experience causes a higher rating.


## 3. Experience Patterns Among Best-Received Titles

Titles are ranked using an adjusted rating that combines rating quality with vote volume. The top 20% within movies and TV are then compared with all eligible titles to see which experiences appear more often among the best-received titles.


In [4]:
# Find experiences that are overrepresented among the best-received titles.

successful_patterns = con.sql(
    load_sql("04_successful_experience_patterns.sql")
).df()

successful_patterns.groupby("media_type").head(10)


,media_type,experience,category,top_title_pct,overall_pct,overrepresentation_pp
0,movie,Unforgettable,After-effect,53.9,24.7,29.2
1,movie,Full of life,Overall experience,49.2,24.6,24.7
2,movie,Epic / grand,Story / world experience,45.6,24.6,21.0
3,movie,Immersive,Overall experience,45.2,24.6,20.6
4,movie,Nostalgic,Connection,45.0,24.9,20.1
5,movie,Life-affirming,Overall experience,43.2,24.6,18.6
6,movie,Intense,Overall experience,42.6,24.6,18.0
7,movie,Fulfilled,After-effect,40.3,24.7,15.6
8,movie,Romantic,Emotion,40.0,24.7,15.3
9,movie,Provokes curiosity,Thinking / engagement,38.8,24.5,14.3


In [5]:
# Put the successful-title pattern beside the weighted rating lift.

pattern_evidence = successful_patterns.merge(
    rating_lift[
        [
            "media_type",
            "experience",
            "weighted_rating_lift"
        ]
    ],
    on=["media_type", "experience"]
)

pattern_evidence.groupby("media_type").head(10)


,media_type,experience,category,top_title_pct,overall_pct,overrepresentation_pp,weighted_rating_lift
0,movie,Unforgettable,After-effect,53.9,24.7,29.2,0.98
1,movie,Full of life,Overall experience,49.2,24.6,24.7,0.88
2,movie,Epic / grand,Story / world experience,45.6,24.6,21.0,0.69
3,movie,Immersive,Overall experience,45.2,24.6,20.6,0.72
4,movie,Nostalgic,Connection,45.0,24.9,20.1,0.61
5,movie,Life-affirming,Overall experience,43.2,24.6,18.6,0.70
6,movie,Intense,Overall experience,42.6,24.6,18.0,0.71
7,movie,Fulfilled,After-effect,40.3,24.7,15.6,0.66
8,movie,Romantic,Emotion,40.0,24.7,15.3,0.52
9,movie,Provokes curiosity,Thinking / engagement,38.8,24.5,14.3,0.60


This combines two signals for each experience: how often it appears among the best-received titles and its vote-weighted rating lift.

## 4. Recommendation Candidates

Recommendations are built for all 55 experiences rather than only the strongest business patterns.

A candidate must:

- score at least 90 for the requested experience,
- fall between the 60th and 80th percentile for adjusted reception,
- and have at least the 25th-percentile vote count for its media type.

The top five candidates for each experience are then ranked by experience score, with adjusted reception used as a tie-breaker.


In [6]:
# Build the candidate pool from SQL.

recommendation_candidates = con.sql(
    load_sql("05_recommendation_candidates.sql")
).df()

# Rank the strongest experience matches first.
top_recommendations = (
    recommendation_candidates
    .sort_values(
        [
            "media_type",
            "experience",
            "experience_score",
            "adjusted_rating"
        ],
        ascending=[True, True, False, False]
    )
    .groupby(
        ["media_type", "experience"],
        as_index=False
    )
    .head(5)
)


In [7]:
# Show a sample of the recommendation output.

top_recommendations[
    [
        "media_type",
        "experience",
        "title",
        "experience_score",
        "vote_average",
        "vote_count",
        "adjusted_rating",
        "reception_percentile"
    ]
].head(25)


,media_type,experience,title,experience_score,vote_average,vote_count,adjusted_rating,reception_percentile
0,movie,Adventurous,Everest,99.9,6.826,5325,6.77,71.8
1,movie,Adventurous,Journey to the Center of the Earth,99.8,6.896,525,6.64,61.3
2,movie,Adventurous,Jungle,99.7,6.705,2224,6.65,61.3
3,movie,Adventurous,The Aeronauts,99.5,6.805,1221,6.67,64.3
4,movie,Adventurous,The Edge,98.9,6.861,1353,6.71,67.2
84,movie,Bittersweet,Chemical Hearts,99.8,7.305,1168,6.91,79.8
85,movie,Bittersweet,Life Itself,99.8,7.169,644,6.76,70.7
86,movie,Bittersweet,Blue Valentine,99.5,6.971,3593,6.86,77.2
87,movie,Bittersweet,Biutiful,99.3,7.245,1239,6.90,79.0
88,movie,Bittersweet,The Light Between Oceans,99.2,7.063,1443,6.82,75.0


## 5. Export Analysis Results

Export the final analysis tables for use in Power BI.

In [8]:
# Set up the folder for Power BI data.

POWERBI_DIR = Path("../powerbi/data")
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
# Combine the experience pattern and rating results into one table.

experience_insights = successful_patterns.merge(
    rating_lift[
        [
            "media_type",
            "experience",
            "category",
            "strong_title_count",
            "strong_avg_rating",
            "other_avg_rating",
            "strong_weighted_rating",
            "other_weighted_rating",
            "title_rating_lift",
            "weighted_rating_lift"
        ]
    ],
    on=["media_type", "experience", "category"]
)

experience_insights.head()

,media_type,experience,category,top_title_pct,overall_pct,overrepresentation_pp,strong_title_count,strong_avg_rating,other_avg_rating,strong_weighted_rating,other_weighted_rating,title_rating_lift,weighted_rating_lift
0,movie,Unforgettable,After-effect,53.9,24.7,29.2,1600.0,7.20,6.32,7.71,6.73,0.88,0.98
1,movie,Full of life,Overall experience,49.2,24.6,24.7,1591.0,7.12,6.34,7.64,6.76,0.78,0.88
2,movie,Epic / grand,Story / world experience,45.6,24.6,21.0,1596.0,7.03,6.37,7.47,6.78,0.66,0.69
3,movie,Immersive,Overall experience,45.2,24.6,20.6,1597.0,7.01,6.38,7.50,6.78,0.63,0.72
4,movie,Nostalgic,Connection,45.0,24.9,20.1,1611.0,7.08,6.35,7.48,6.87,0.73,0.61


In [10]:
# Export the final tables used in the dashboard.

catalog = con.sql("""
    SELECT
        title_key,
        media_type,
        title,
        release_date,
        vote_average,
        vote_count,
        popularity
    FROM titles
""").df()

experience_insights.to_csv(
    POWERBI_DIR / "experience_insights.csv",
    index=False
)

top_recommendations.to_csv(
    POWERBI_DIR / "recommendations.csv",
    index=False
)

catalog.to_csv(
    POWERBI_DIR / "catalog.csv",
    index=False
)

print("Power BI files exported.")

Power BI files exported.


In [11]:
# Confirm the files were created.

list(POWERBI_DIR.iterdir())

[WindowsPath('../powerbi/data/catalog.csv'),
 WindowsPath('../powerbi/data/experience_insights.csv'),
 WindowsPath('../powerbi/data/recommendations.csv')]

In [12]:
con.close()
